In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML


class Ball(object):
    '''a ball
    '''
    def __init__(self,r,v,m,radius): #initialize the object, including the radius
        self.r=np.array(r)
        self.v=np.array(v)
        self.m=m
        self.radius = radius
        self.path={'x':[r [0]],'y':[r[1]]} #self.path is a dictionry, containing two keys 'x', 'y', the value of each key is a list
        self.graph=plt.Circle((self.path['x'][0], self.path['y'][0]),radius=self.radius,zorder=2)
        #r,v,m,path are attributes of ball
    def record(self):
        self.path['x'].append(self.r[0]) #append the current x position to path
        self.path['y'].append(self.r[1]) #append the current y position to path
    def draw(self,ax):
        ax.add_patch(self.graph)
    def update_position(self,indx):
        self.graph.center=self.path['x'][indx], self.path['y'][indx]
    def undraw(self):
        self.graph.remove()


def trajectory(frame_indx,ball,factor):
    indx = factor*frame_indx
    for i_ball in range(len(ball)):
        ball[i_ball].update_position(indx)

def motion_billiard(dt, num_step, ball, length, width, t=0):
    def boundary(ball):
        '''checking if the ball is within the boundary
        '''
        if ball.r[0]-ball.radius<0 or ball.r[0] + ball.radius>length:
            ball.v[0]=-ball.v[0]
        if ball.r[1]-ball.radius<0 or ball.r[1] + ball.radius>width:
            ball.v[1]=-ball.v[1]
        return ball
    time_list=[t]
    num_ball = len(ball)
    #starting iteration
    for i_step in range(num_step):
        for i_ball in range(num_ball):
            for j_ball in range(i_ball+1,num_ball):
                dr_vec = ball[i_ball].r - ball[j_ball].r
                dr_nrm = np.linalg.norm(dr_vec)
                if dr_nrm <= ball[i_ball].radius + ball[j_ball].radius: #collision
                    n_hat = dr_vec/dr_nrm
                    #decompose vi
                    vi_n = np.dot(ball[i_ball].v, n_hat)*n_hat #(v.n_hat)*n_hat
                    vi_t = ball[i_ball].v - vi_n  #v=v_n + v_t
                    #decompose vj
                    vj_n = np.dot(ball[j_ball].v, n_hat)*n_hat
                    vj_t = ball[j_ball].v - vj_n  #v=v_n + v_t
                    m_i = ball[i_ball].m
                    m_j = ball[j_ball].m
                    v_center= (vi_n*m_i + vj_n*m_j)/(m_i+m_j) #velocity of the center of mass
                    vi_n_new = 2*v_center-vi_n  #normal velocities after collision
                    vj_n_new = 2*v_center-vj_n

                    ball[i_ball].v = vi_t + vi_n_new
                    ball[j_ball].v = vj_t + vj_n_new

        ##check boundary
        for i_ball in range(num_ball):
            ball[i_ball] = boundary(ball[i_ball])

       ##update position
        t=t+dt
        time_list.append(t)
        for i_ball in range(num_ball):
            ball[i_ball].r += ball[i_ball].v*dt
            ball[i_ball].record()
    #Visualization
    fig, ax = plt.subplots(figsize=(8,8*width/length))
    ax.plot([0,0],[0,width],'g')
    ax.plot([length,length],[0,width],'g')
    ax.plot([0,length],[0,0],'g')
    ax.plot([0,length],[width,width],'g')
    ax.axis('equal')
    ax.axis('off');
    for i_ball in range(num_ball):
        ball[i_ball].draw(ax)
    plt.close()
    num_frame = int(len(ball[0].path['x'])/100) # the total number of frames
    anim = animation.FuncAnimation(fig, lambda x: trajectory(x,ball,100), frames=num_frame, interval=40)
    return anim


In [8]:
ball1 = Ball(r=[5.0,6.0], v=[-1,1.0], radius=2, m=1.0)
ball2 = Ball(r=[8.0,2.0], v=[-1,3.1], radius=1, m=1.0)
ball3 = Ball(r=[14.0,8.0], v=[-1,3.1], radius=1, m=1.0)
anim = motion_billiard(dt=0.001, num_step=20000, ball=[ball1,ball2,ball3], length=20, width=10)
HTML(anim.to_html5_video())